### Challenge 1:
Tasks:

- *Fill missing salary with the average salary per department.*
- *Convert hire_date to date type.*
- *Add a column years_with_company based on today's date.*
- *Filter only employees who have been with the company for 3+ years.*
- *Show average salary by department.*

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window
from pyspark.sql.types import *

In [0]:
employee_df = spark.createDataFrame([
    (101, "Alice", "HR", 58000, "2019-04-01"),
    (102, "Bob", "Engineering", 92000, "2020-07-15"),
    (103, "Charlie", "Sales", None, "2018-02-20"),
    (104, "Diana", "Engineering", 99000, None),
    (105, "Evan", "HR", 61000, "2021-09-12")
], ["emp_id", "name", "department", "salary", "hire_date"])

In [0]:
employee_df.show()

In [0]:
# Fill missing salary with the average salary per department.
# Convert hire_date to date type.
# Add a column years_with_company based on today's date.
# Filter only employees who have been with the company for 3+ years.
# Show average salary by department.

employee_df = (
    employee_df
    .withColumn("salary",
                 when(col("salary").isNull(),
                 avg("salary")
                 .over(Window.partitionBy("department")))
                 .otherwise(col("salary"))
            )
    .withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))
    .withColumn("years_with_company", 
                round(months_between(current_date(), col("hire_date")) / lit(12), 0)
                .cast(IntegerType())
            )
    .filter(col("years_with_company") >= 3)
    .groupBy("department")
    .agg(avg("salary").alias("avg_salary"))
    )

In [0]:
employee_df.show()
employee_df.printSchema()

### Challenge 2:

Tasks:

- Clean the price column by removing $ and converting to float.
- Standardize in_stock values to boolean (True/False).
- Fill null prices with the average price.
- Add a new column category based on product name:
- "Laptop", "Tablet", "Phone" → "Mobile Devices"
- "Monitor", "Keyboard" → "Accessories"
- Show sorted products by price descending.



In [0]:
product_df = spark.createDataFrame([
    ("A001", "Laptop", "$999.99", "Yes"),
    ("A002", "Phone", "499.5", "No"),
    ("A003", "Tablet", "$299", "yes"),
    ("A004", "Monitor", "199.99", "NO"),
    ("A005", "Keyboard", None, "Yes")
], ["product_id", "name", "price", "in_stock"])

In [0]:
product_df.show()

In [0]:
# Clean the price column by removing $ and converting to float.
# Standardize in_stock values to boolean (True/False).
# Fill null prices with the average price.
# Add a new column category based on product name:
# "Laptop", "Tablet", "Phone" → "Mobile Devices"
# "Monitor", "Keyboard" → "Accessories"
# Show sorted products by price descending.

window_spec = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

product_df = (
    product_df
    .withColumn("price", regexp_replace(col("price"), "[$]", "").cast("float"))
    .withColumn("in_stock", when(upper(col("in_stock")) == "YES", True).otherwise(False))
    .withColumn("price", when(
        col("price").isNull(),
        avg("price").over(window_spec)
        ).otherwise(col("price"))
    )
    .withColumn("price", round(col("price"), 2))
    .withColumn("category", when(col("name").isin("Laptop", "Tablet", "Phone"), "Mobile Devices")
                .when(col("name").isin("Monitor", "Keyboard"), "Accessories")
            )
    .sort(col("price").desc())
)

In [0]:
product_df.show()
product_df.printSchema()

### Challenge 3

Tasks:
- Convert timestamp to timestamp type.
- Extract endpoint path (e.g., /home, /cart) to a new column endpoint.
- Add a column is_error if status is 400 or above.
- Count how many requests each IP made.
- Show the number of error requests per endpoint.

In [0]:
df_log = spark.createDataFrame([
    ("2025-03-01 12:00:00", "192.168.1.1", "GET /home", 200),
    ("2025-03-01 12:01:00", "192.168.1.2", "POST /login", 401),
    ("2025-03-01 12:02:00", "192.168.1.3", "GET /products", 200),
    ("2025-03-01 12:03:00", "192.168.1.1", "GET /cart", 500),
    ("2025-03-01 12:04:00", "192.168.1.4", "GET /home", 404)
], ["timestamp", "ip", "request", "status"])

In [0]:
df_log.show()

In [0]:
# Convert timestamp to timestamp type.
# Extract endpoint path (e.g., /home, /cart) to a new column endpoint.
# Add a column is_error if status is 400 or above.

df_log = (
    df_log
    .withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("endpoint", split(col("request"), " ")[1])
    .withColumn("is_error", when(col("status") > 400, True).otherwise(False))
)

# Count how many requests each IP made.
df_requests = (
    df_log
    .groupBy("ip")
    .agg(count("*").alias("requests"))
    .orderBy(col("requests").desc())
)

# Show the number of error requests per endpoint.
df_error = (
    df_log
    .groupBy("endpoint")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
)

df_log.show()
df_requests.show()
df_error.show()


### Challenge 4

- Convert event_time to timestamp.
- Filter only "add_to_cart" and "purchase" events.
- Create a column session_minute = minute of the event (use minute() function).
- Create a column is_conversion = True if event is "purchase", else False.
- Group by user_id and count how many conversions and add-to-cart events each user made.
- Sort users by most conversions, then most add-to-cart.

In [0]:
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("product_id", StringType(), True),
])


df_clickstream = spark.createDataFrame([
    ("u001", "2025-04-12 14:23:54", "page_view", "/home", None),
    ("u001", "2025-04-12 14:25:00", "add_to_cart", "/product/123", "p123"),
    ("u001", "2025-04-12 14:30:10", "purchase", "/checkout", "p123"),
    ("u002", "2025-04-12 14:10:45", "page_view", "/home", None),
    ("u002", "2025-04-12 14:12:00", "add_to_cart", "/product/456", "p456"),
    ("u002", "2025-04-12 14:50:00", "page_view", "/cart", None),
    ("u003", "2025-04-12 14:00:00", "add_to_cart", "/product/789", "p789"),
    ("u003", "2025-04-12 14:05:00", "purchase", "/checkout", "p789"),
    ("u004", "2025-04-12 15:00:00", "page_view", "/home", None),
    ("u004", "2025-04-12 15:02:00", "add_to_cart", "/product/123", "p123")
], schema)

In [0]:
df_clickstream.show()
df_clickstream.printSchema()


In [0]:
# Convert event_time to timestamp.
# Filter only "add_to_cart" and "purchase" events.
# Create a column session_minute = minute of the event (use minute() function).
# Create a column is_conversion = True if event is "purchase", else False.
# Group by user_id and count how many conversions and add-to-cart events each user made.
# Sort users by most conversions, then most add-to-cart.

df_clickstream = (
    df_clickstream
    .withColumn("event_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss"))
    .filter(col("event_type").isin("add_to_cart", "purchase"))
    .withColumn("session_minute", minute(col("event_time")))
    .withColumn("is_conversion", when(col("event_type") == "purchase", True).otherwise(False))
    .groupBy("user_id")
    .agg(count(when(col("is_conversion") == True, True)).alias("conversions"),
          count(when(col("event_type") == "add_to_cart", True)).alias("add_to_cart"))
)

In [0]:
df_clickstream.show()
df_clickstream.printSchema()